### Acceptance Criteria
 - At least 3 JSON files land in /Volumes/dev_<you>/ops/landing/trips/, each a different slice of samples.nyctaxi.trips
 - The same data also exists as CSV and Parquet in sibling directories
 - One JSON file contains an extra column the others don't
 - One JSON file contains at least 5 deliberately bad records (nulls in key fields, negative fares)
 - You can list the files with dbutils.fs.ls and see distinct file sizes
### Subtasks
 - Export batch 1 — Read samples.nyctaxi.trips, limit() a few thousand rows, write JSON to the volume. Hint: .write.format("json").save(save) writes a directory of part-files; that's fine and realistic.
 - Export batches 2 and 3 — Use a different WHERE filter each time so the batches are distinct. Hint: filter on tpep_pickup_datetime ranges.
 - Create the schema-drift file — Add a computed column (e.g. fare_per_mile) before writing batch 4. This is LAB-03's evolution trigger.
 - Create the dirty file — union some rows with nulls and negatives. Hint: build them with spark.createDataFrame — remember the 128 MB local row-size cap on serverless.
 - Repeat for CSV and Parquet — Different directories. Hint: note which formats infer types and which don't; that's the LAB-03 lesson.
 - Verify — dbutils.fs.ls each directory.
#### Note
Keep this notebook. Later labs will re-run it to simulate new data arriving.

### Change Log

[2026-09-05] v.0.0.1
- Add initial implementation of extract for files from table for further labs  
- Finish implementation in cell 7: ETL
- TODO: Fix bugs, syntax stupidity was used in Cell 7, and need to be fixed (sorry, I got a bit drunk!)

[2026-09-06] v0.0.2
- Fix bugs, syntax issues, add test verification and dirty files scenario
- Execute the notebook to generate the files
- Lab02 DONE -> There might be a chance batch with dirty files to have ONLY dirty rows!

# Batches
1. tpep_pickup_date >= 2016-01-01 and tpep_pickup_date <= 2016-01-04 - 1624 records  
2. tpep_pickup_date >= 2016-01-05 and tpep_pickup_date <= 2016-01-08 - 1160 records  
3. tpep_pickup_date >= 2016-01-09 and tpep_pickup_date <= 2016-01-12 - 1460 records  
4. tpep_pickup_date >= 2016-01-13 and tpep_pickup_date <= 2016-01-15 - 787 records  
    a. Schema Drift Batch
5. tpep_pickup_date >= 2016-01-16 and tpep_pickup_date <= 2016-01-16 - limit 50 records  
    a. Dirty File - these 50 records will be unioned with dirty records having null and negative values

In [0]:
from datetime import datetime

from pyspark.sql.functions import col, lit, try_divide, nvl

# JSON, CSV, Parquet Formats

In [0]:
formats: list = ["json", "csv", "parquet"]
etl_configuration: dict = {
    "1": {
        "low_watermark": "2016-01-01",
        "high_watermark": "2016-01-04",
        "special_case": None
    },
    "2": {
        "low_watermark": "2016-01-05",
        "high_watermark": "2016-01-08",
        "special_case": None
    },
    "3": {
        "low_watermark": "2016-01-09",
        "high_watermark": "2016-01-12",
        "special_case": None
    },
    "4": {
        "low_watermark": "2016-01-13",
        "high_watermark": "2016-01-15",
        "special_case": "schema_drift"
    },
    "5": {
        "low_watermark": "2016-01-16",
        "high_watermark": "2016-01-16",
        "special_case": "dirty_files"
    },
}

In [0]:
nyctaxi_trips_table = spark.table("samples.nyctaxi.trips")

for format_type in formats:
    print(f"####### Start processing of file type {format_type} #######")
    for batch_number, config in etl_configuration.items():
        target_path: str = f"/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_{format_type}/batch_{batch_number}"
        print(f"###### Start Batch {batch_number}! ######\n")
        print(f"Config: \n  {config} ")
        if config.get("special_case") is None:
            (
                nyctaxi_trips_table.filter(
                    (col("tpep_pickup_datetime") >= lit(config.get("low_watermark", None)).cast("date"))
                    & (col("tpep_pickup_datetime") <= lit(config.get("high_watermark", None)).cast("date")) 
                )
                .write.format(format_type).mode("overwrite")
                .save(target_path)
            )
        elif config.get("special_case", None) == "schema_drift":
            (
                nyctaxi_trips_table.filter( 
                    (col("tpep_pickup_datetime") >= lit(config.get("low_watermark", None)).cast("date"))
                    & (col("tpep_pickup_datetime") <= lit(config.get("high_watermark", None)).cast("date"))
                ).withColumn(
                    "fare_per_mile", 
                    nvl(try_divide(col("fare_amount"), col("trip_distance")).cast("decimal(19,2)"), lit(0))
                )
                .write.format(format_type).mode("overwrite")
                .save(target_path)
            )
        elif config.get("special_case", None) == "dirty_files":
            schema = nyctaxi_trips_table.schema          # guarantees UNION compatibility
            cols = [f.name for f in schema.fields]
            print(cols)                  # confirm order before trusting the tuples below

            ts = lambda s: datetime.strptime(s, "%Y-%m-%d %H:%M:%S")

            # Order must match `cols` exactly:
            # (pickup_ts, dropoff_ts, trip_distance, fare_amount, pickup_zip, dropoff_zip)
            dirty_rows = [
                (ts("2016-01-16 10:00:00"), ts("2016-01-16 10:20:00"),  4.1,     -12.50, None, None),
                (ts("2016-01-16 11:00:00"), ts("2016-01-16 11:05:00"),  0.0,       8.00, None, 10012),
                (ts("2016-01-16 12:00:00"), ts("2016-01-16 12:30:00"),  6.7,      22.30,  None, 10013),
                (ts("2016-01-16 14:00:00"), ts("2016-01-16 13:00:00"),  3.3,      15.00, 10003, None),
                (None,                      None,                       2.2,      11.00, None, 10015),
                (ts("2016-01-16 15:00:00"), ts("2016-01-16 15:10:00"),  1.1,  999999.99, None, 10016),
                (ts("2016-01-16 16:00:00"), ts("2016-01-16 16:25:00"), -3.2,      18.40, 10006, None),
                (ts("2016-01-16 17:00:00"), ts("2016-01-16 18:00:00"), 45.0,       0.00, 10007, 10018),
                (None,                      None,                      None,       None,  None, 10019),
                (ts("2016-01-16 19:00:00"), ts("2016-01-16 19:15:00"),  2.8,      13.20, None,    -1),
            ]

            dirty_df = spark.createDataFrame(dirty_rows, schema=schema)

            clean_df = nyctaxi_trips_table.filter( 
                (col("tpep_pickup_datetime") >= lit(config.get("low_watermark", None)).cast("date")) 
                & (col("tpep_pickup_datetime") <= lit(config.get("high_watermark", None)).cast("date"))
            ).orderBy(col("tpep_pickup_datetime")).limit(50)

            (
                clean_df.unionByName(dirty_df)
                .write.format(format_type).mode("overwrite")
                .save(target_path)
            )
        else:
            raise ValueError(f"Invalid special case: {config.get('special_case')}")
        print(f"###### Start Batch {batch_number}! ######\n")

In [0]:
for f in ["json", "csv", "parquet"]:
    for b in ["1", "2", "3", "4", "5"]:
        print("###### Start check for file type {f} batch {b}")
        print(f, b, len(dbutils.fs.ls(f'/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_{f}/batch_{b}')))
        print("###### End check for file type {f} batch {b}")
        